In [1]:
import os

In [2]:
os.getcwd()


'/Users/rachnaraj/Documents/Research/Winter25/ExtractorProject/RepoClonerAnalyser/Utility'

In [7]:
import pandas as pd

In [17]:
import pandas as pd

# Load the CSV file
df = pd.read_csv("Dataset/BUMP_with_github.csv")

# Define invalid values
invalid_values = ["Artifact Not Found", "No GitHub Repo Found", None, ""]

# Filter rows where GithubURL is valid
df_filtered = df[~df["githubURL"].isin(invalid_values)]

# Save the cleaned file
df_filtered.to_csv("Dataset/BUMP_valid_github.csv", index=False)

print(f"Filtered CSV saved as x_valid_github.csv with {len(df_filtered)} valid rows.")


Filtered CSV saved as x_valid_github.csv with 170 valid rows.


In [18]:
log_file_path = "/Volumes/Rachna-HD/ClonedRepo/ClonedRepo/Clients/clone_errors.txt"

In [20]:
if os.path.exists(log_file_path):
    print("✅ Log file found.")
else:
    print("❌ Log file NOT found. Check the path.")

✅ Log file found.


In [22]:
import pandas as pd
import re

# STEP 1: Load the original CSV
df = pd.read_csv("Dataset/BUMP_valid_github.csv")
log_file_path = "/Volumes/Rachna-HD/ClonedRepo/ClonedRepo/Clients/clone_errors.txt"

# Parse the log file for failed GitHub clone URLs
failed_urls = set()
with open(log_file_path, "r") as f:
    for line in f:
        match = re.match(r"(https://github.com/.*?\.git): Cloning into", line.strip())
        if match:
            failed_urls.add(match.group(1).strip())

print(failed_urls)
# STEP 3: Filter out rows where GithubURL is in the failed list
df_filtered = df[~df["url"].isin(failed_urls)]

# STEP 4: Save the result
df_filtered.to_csv("Dataset/BUMP_valid_github.csv", index=False)

print(f"Filtered CSV saved with {len(df_filtered)} rows (removed {len(df) - len(df_filtered)} failed clones).")


{'https://github.com/artipie/http.git', 'https://github.com/codehaus-plexus/plexus-archiver.git'}
Filtered CSV saved with 170 rows (removed 0 failed clones).


In [23]:
import pandas as pd
import re

# STEP 1: Load the original CSV
df = pd.read_csv("Dataset/BUMP_valid_github.csv")
log_file_path = "/Volumes/Rachna-HD/ClonedRepo/ClonedRepo/Library/clone_errors.txt"

# Parse the log file for failed GitHub clone URLs
failed_urls = set()
with open(log_file_path, "r") as f:
    for line in f:
        match = re.match(r"(https://github.com/.*?\.git): Cloning into", line.strip())
        if match:
            failed_urls.add(match.group(1).strip())

print(failed_urls)
# STEP 3: Filter out rows where GithubURL is in the failed list
df_filtered = df[~df["githubURL"].isin(failed_urls)]

# STEP 4: Save the result
df_filtered.to_csv("Dataset/BUMP_valid_github.csv", index=False)

print(f"Filtered CSV saved with {len(df_filtered)} rows (removed {len(df) - len(df_filtered)} failed clones).")


set()
Filtered CSV saved with 170 rows (removed 0 failed clones).


In [25]:
import pandas as pd
import os
from urllib.parse import urlparse

# === CONFIGURATION ===
csv_path = "Dataset/BUMP_valid_github.csv"  # Your filtered CSV file
client_repo_base_path = "/Volumes/Rachna-HD/ClonedRepo/ClonedRepo/Clients"  # Where you cloned all repos
output_csv_path = "Dataset/BUMP_valid_github_with_clone_status.csv"

# === LOAD CSV ===
df = pd.read_csv(csv_path)

# === FUNCTION TO CLEAN GITHUB URL AND GET FOLDER NAME ===
def extract_repo_name(github_url):
    if pd.isna(github_url):
        return None
    base_url = github_url.split("/pull")[0].split("/issues")[0].rstrip("/")
    parts = urlparse(base_url).path.strip("/").split("/")
    if len(parts) >= 2:
        return parts[1]  # folder is usually named after the repo name (not full org/repo)
    return None

# === CHECK IF FOLDER EXISTS ===
def check_clone_status(github_url):
    repo_name = extract_repo_name(github_url)
    if not repo_name:
        return "Cloning Error"
    
    folder_path = os.path.join(client_repo_base_path, repo_name)
    if os.path.isdir(folder_path):
        return "Cloned"
    else:
        return "Cloning Error"

# === APPLY THE CHECK TO EACH ROW ===
df["CloneStatus"] = df["clientGithubURL"].apply(check_clone_status)

# === SAVE THE OUTPUT ===
df.to_csv(output_csv_path, index=False)

print(f"Cloning status checked and saved to: {output_csv_path}")


Cloning status checked and saved to: Dataset/BUMP_valid_github_with_clone_status.csv


In [29]:
import pandas as pd
import os
from urllib.parse import urlparse

# === CONFIGURATION ===
csv_path = "Dataset/BUMP_valid_github.csv"
output_csv_path = "Dataset/BUMP_valid_github_with_clone_status.csv"

client_repo_base_path = "/Volumes/Rachna-HD/ClonedRepo/ClonedRepo/Clients"
library_repo_base_path = "/Volumes/Rachna-HD/ClonedRepo/ClonedRepo/Library"

# === LOAD CSV ===
df = pd.read_csv(csv_path)

# === HELPER FUNCTION ===
def extract_repo_name(github_url):
    if pd.isna(github_url):
        return None
    base_url = github_url.split("/pull")[0].split("/issues")[0].rstrip("/")
    parts = urlparse(base_url).path.strip("/").split("/")
    if len(parts) >= 2:
        return parts[1]  # repo name (e.g., dropwizard-pac4j)
    return None

def check_clone_status(github_url, base_path):
    repo_name = extract_repo_name(github_url)
    if not repo_name:
        return "Cloning Error"
    folder_path = os.path.join(base_path, repo_name)
    return "Cloned" if os.path.isdir(folder_path) else "Cloning Error"

# === APPLY BOTH CHECKS ===
df["ClientCloneStatus"] = df["clientGithubURL"].apply(lambda url: check_clone_status(url, client_repo_base_path))
df["LibraryCloneStatus"] = df["libraryGithubURL"].apply(lambda url: check_clone_status(url, library_repo_base_path))

# === SAVE THE RESULT ===
df.to_csv(output_csv_path, index=False)
print(f" Clone status columns added and saved to: {output_csv_path}")


 Clone status columns added and saved to: Dataset/BUMP_valid_github_with_clone_status.csv


In [30]:
import os
import pandas as pd
import re
from collections import Counter

# === CONFIGURATION ===
csv_path = "Dataset/FinalBUMP_Instances.csv"
library_repo_base_path = "/Volumes/Rachna-HD/ClonedRepo/ClonedRepo/Library"
output_csv_path = "Dataset/FinalBUMP_Instances.csv"

# === LOAD CSV ===
df = pd.read_csv(csv_path)

# === GET repo folder name from GitHub URL ===
def get_repo_folder_name(github_url):
    if pd.isna(github_url):
        return None
    base_url = github_url.split("/pull")[0].split("/issues")[0].rstrip("/")
    return base_url.split("/")[-1]  # e.g., 'jackson-databind'

# === Extract most common top-level package from .java files ===
def extract_actual_import_base(repo_name):
    repo_path = os.path.join(library_repo_base_path, repo_name)
    if not os.path.isdir(repo_path):
        return "Cloning Error"

    package_names = []
    for root, _, files in os.walk(repo_path):
        for file in files:
            if file.endswith(".java"):
                file_path = os.path.join(root, file)
                try:
                    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                        for line in f:
                            line = line.strip()
                            if line.startswith("package "):
                                match = re.search(r'package\s+([\w\.]+);', line)
                                if match:
                                    pkg = match.group(1)
                                    if len(pkg.split(".")) >= 2:
                                        package_names.append(".".join(pkg.split(".")[:3]))
                                break  # only need first valid package line per file
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")

    if package_names:
        most_common = Counter(package_names).most_common(1)
        return most_common[0][0] if most_common else None
    else:
        return "No package found"

# === PROCESS EACH ROW ===
df["LibraryRepoFolder"] = df["libraryGithubURL"].apply(get_repo_folder_name)
df["ActualImportBase"] = df["LibraryRepoFolder"].apply(extract_actual_import_base)

# === SAVE ===
df.to_csv(output_csv_path, index=False)
print(f"Done. Saved with ActualImportBase to: {output_csv_path}")


Done. Saved with ActualImportBase to: Dataset/FinalBUMP_Instances.csv
